# Broad cohort: champion proficiency versus matchup, before and after personalization

**Question:** When does recorded champion proficiency outweigh a better matchup, and how does accounting for the individual player change the recommendation?

This notebook prepares the completed 300-player Bronze–Diamond cohort, builds externally defined A–J mastery-payoff groups, and specifies matched shared/grouped-experience hierarchical logistic models. The Master cohort remains separate for a later comparison. There is no synergy analysis.

**Default: preparation/EDA only.** `RUN_MODELING=False` prevents optimization, sampling, evaluation, and model-based recommendations. No Riot or LoLalytics collection is invoked. Prepared outputs retain all 5,706 played rows and 47,356 eligible alternatives.

## 1. Imports, pinned inputs, and the single fitting switch

Use the repository's environment with pandas, numpy, scipy, scikit-learn, matplotlib, openpyxl, pyarrow, and IPython. Fitting additionally requires torch and **pyro-ppl==1.9.2**. Nothing is installed automatically; temporary backend paths are not loaded automatically.

The source is the completed matchup export `20261027T194148939685Z_3b8d192e`. Metadata and file hashes are checked before use. The workbook is pinned separately. Changing either source requires an explicit review/re-pin and creates a new analysis version.

Run All first with the switch left False. For a small execution check later, set `profile='smoke'`, then enable the switch. Smoke uses the same NUTS likelihood on at most 24 training matches, 2 chains, 25 warmup + 25 retained draws; it never authorizes substantive results. Full mode uses 4 sequential chains × 600 warmup + 1,200 retained draws **per model**, with maximum tree depth 9 and target acceptance 0.9. This can be expensive; it has not been run during implementation.

In [ ]:
from pathlib import Path
import sys, json, importlib, importlib.util
import numpy as np
import pandas as pd
from IPython.display import display, Image

start = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [start, *start.parents]
    if (p / 'hlr_model/broad_rank_pipeline.py').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open from the repository root or hlr_model/.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from hlr_model import broad_rank_modeling as modeling
importlib.reload(modeling)

RUN_MODELING = False
CONFIG = modeling.Config(profile='full')
print('Fitting enabled:', RUN_MODELING)
print('Sampling profile:', CONFIG.profile, CONFIG.sampling)
print('Pyro installed in this notebook environment:', importlib.util.find_spec('pyro') is not None)
print('No collector or network request is used by this notebook.')

## 2. Read the completed source and inspect actual schemas

The source metadata report 28,876 published of 28,936 requested unique benchmarks, with 60 not published in verified tables. Availability is not equivalent to a failed request. Missing raw probabilities remain missing. The 100-game flag is reporting only.

Source row identifiers, benchmark evidence, raw/adjusted values, candidate mappings, and initial-history metadata remain intact. Raw Global exact-tier probability enters the model with an estimated slope; the derived tier-centered percentage is neither a probability nor a fixed logit offset.

In [ ]:
bundle = modeling.load_source(PROJECT_ROOT, CONFIG)
print('Selected completed snapshot:', bundle['path'].relative_to(PROJECT_ROOT))
display(pd.DataFrame([
    {'table': name, 'rows': len(frame), 'columns': len(frame.columns)}
    for name, frame in bundle['frames'].items()
]))
display(bundle['frames']['matchup_benchmark_lookup'].status.value_counts().rename('unique_matchups').to_frame())
print('Played benchmark availability:')
display(bundle['frames']['played_target_games'].benchmark_status.value_counts().rename('rows').to_frame())

## 3. Inspect the external workbook and derive A–J groups

The workbook contains **Data Intro**, **Easiest to Learn**, and **Best to Master**. Only the full **Best to Master** table's `Medium Mastery Win Rate` and `High Mastery Win Rate` define these groups. The source's ratio column is preserved and is **not** used instead of a difference.

Excel cells store probabilities and use percent formatting. Thus:

- `raw_gain_pp = 100 × (high_probability − medium_probability)`
- `grouping_score = max(0, raw_gain_pp)`
- J contains every usable row with score exactly zero.
- Positive scores are sorted descending; A–I use nine equal rank intervals. A tied block receives its mean zero-based rank r, then group index `min(8, floor(9*r/(N_positive−1)))`. A single positive row goes to A. Whole ties stay together; uneven/empty groups are reported.
- Unusable/missing rows stay Unclassified, including entries marked “low data.” There is no median imputation.

Stable IDs use the project's explicit Riot-name/display-alias registry, with the reviewed workbook name **Nunu & Willump → Nunu (20)**. Unknown/ambiguous names stop grouping rather than guessing.

The original workbook, all original rates, signed gains, clipped scores, classifications, row/sheet provenance, intro text, source URL, and hashes are preserved. The study uses patch **13.7**, mostly Gold, roughly equal EUW/NA/KR samples; high mastery means **100K+ mastery points**, medium 10K–100K. These are external grouping labels, not recorded-game counts or universal causal learning curves. Published gains never enter predicted probabilities.

In [ ]:
groups, study_rows, study_metadata = modeling.load_groups(PROJECT_ROOT, CONFIG)
print('Workbook:', CONFIG.study_file)
print('Worksheets:', study_metadata.get('worksheets', []))
print('Study source:', study_metadata.get('source_url', 'missing'))
print('Grouping ready:', study_metadata['modeling_ready'])
if study_metadata['available']:
    print('Study rows:', study_metadata['source_rows'], '| usable:', study_metadata['usable_rows'])
    display(pd.DataFrame({
        'full_study_champions': study_metadata['full_study_group_counts'],
        'registry_champions': study_metadata['registry_group_counts'],
    }).reindex(modeling.GROUPS))
    display(study_rows.loc[~study_rows.study_usable,
        ['study_champion','study_medium_original','study_high_original','classification_reason','mastery_group']])
else:
    print('Missing input:', study_metadata['expected_path'])
print(modeling.TIE_RULE)

## 4. Audit pregame histories, preserve candidates, and freeze chronological partitions

Proficiency is **recorded prior champion games across all roles**. It is not lifetime experience. Initial histories contain up to 170 season-bounded games; they expand with earlier targets completed strictly before kickoff. This notebook independently reconstructs counts, the existing all-role main (most prior games; smallest champion ID breaks ties), historical Beta(1,1) WR, and same-role eligibility.

Alternatives are exactly previously played same-role champions, excluding the actual pick and known opponent. Each alternative gets its **own** prior counts and historical WR; it never receives the target outcome as a training label. At zero recorded champion games, Beta(1,1) WR = 0.5 is a historical prior assumption, not a filled external benchmark. The workbook's most-common-role column does not replace the player's target role.

Preserve the existing Master analysis dates for a later matched comparison:
- Train: kickoff before **2026-10-02 22:25:06.238 UTC**, with outcomes completed before that boundary.
- Validation: that boundary through **2026-10-05 01:32:18.312 UTC**, with outcomes completed before the test boundary.
- Final test: later kickoffs; no held-out refit or settings selection.
- All rows of a shared match stay together. Whole unfinished matches are purged at the corresponding boundary.

Earlier held-out games may update later pregame history after completion, but their outcomes never refit parameters or player effects. Neither specifications nor k are chosen using held-out outcomes. Both models use identical usable rows and partitions. Unclassified and sparse positive-count benchmarks are retained; exclusions and purged rows are explicitly reported.

In [ ]:
prepared = modeling.prepare(bundle, groups, CONFIG)
tables = modeling.coverage_tables(prepared, groups)
print('Rows retained:', {k: len(prepared[k]) for k in ('played','alternatives')})
print('Observed usable data rows:', int(prepared['played'].usable_data.sum()))
print('Unplayed rows with an outcome:', int(prepared['alternatives'].candidate_outcome.notna().sum()))
display(tables['split_support'])
display(tables['data_exclusions'])
print('Initial history distribution, one row per player:')
display(tables['history_depth'].initial_history_games.describe().to_frame())
assert prepared['played'].groupby('match_id').split.nunique().eq(1).all()

## 5. Coverage and recorded experience support

Display distinct players alongside games. Repeated target rows from one player do not establish broad support. Source rows remain in the saved prepared datasets even when they cannot be modeled.

Experience bins are illustrative support summaries only: 0, 1–4, 5–9, 10–19, 20–49, 50+. They are not model cutoffs or replacements for continuous saturating experience.

In [ ]:
display(tables['rank_role_support'])
display(tables['group_support'])
display(tables['experience_distribution'])
display(tables['experience_bins'].pivot(index='mastery_group', columns='experience_bin', values='players').reindex(index=modeling.GROUPS, columns=modeling.BINS))
print('Unclassified champions appearing in played or alternative data:')
display(tables['unclassified_champions'])
display(tables['matchup_coverage'])

## 6. Save versioned preparation, grouping, figures, splits, and model configuration

Outputs are separate from collectors and Master analysis, under `hlr_model/data/analyses/broad_rank_mastery_v1/<version>/`. Identical pinned input/configuration/code reuses the verified preparation version; changed inputs or settings create a new version. Fitting caches individual completed chains inside that version. This lets an interrupted fit resume without overwriting other analyses.

The preparation manifest hashes all prepared outputs. Each model's fit manifest freezes the exact training design, observations, inference settings, backend version, seed, and model source. No fitted result is implied by a preparation manifest.

In [ ]:
OUTPUT_PATH = modeling.prepare_outputs(
    PROJECT_ROOT, bundle, groups, study_rows, study_metadata, prepared, tables, CONFIG)
print('Analysis version:', OUTPUT_PATH.relative_to(PROJECT_ROOT))
print('Prepared data, source/grouping metadata, coverage tables, split assignments and figures saved.')
for name in ('rank_role_support','group_support','experience_and_history','experience_bin_support','matchup_coverage'):
    display(Image(filename=str(OUTPUT_PATH / 'figures' / (name + '.png'))))

## 7. Matched hierarchical logistic specifications

For observed training row i:

`logit P(win_i) = alpha + X_i beta + rank_i + role_i + champion_baseline_i + player_i + player_champion_i + b_group(i) × n_i/(n_i+k)`

X contains training-standardized **raw matchup probability**, **log1p(prior total games)**, and the selected champion's **all-role historical WR Beta(1,1)**. These retain the existing nonlinear tradeoff model's covariates. No external gain, clipped score, tier-centered percentage, current result, or future history is added.

**Shared:** one experience amplitude b. **Grouped:** the same model plus partially pooled A–J/Unclassified amplitude deviations. Both initially share one k across all groups, with the established log(k) ~ Normal(log 30, 1) prior and training-only estimation. There are no per-champion curve shapes or unnecessary rank/group interactions. J may have any fitted effect; Unclassified has one pooled fallback slope with uncertainty, not independent champion-specific curves.

Regularization: fixed continuous coefficients/intercept Normal(0,2); rank/role contrasts Normal(0,1); shared amplitude Normal(0,1); log player/champion scales Normal(log .3,.7); log player–champion scale Normal(log .15,.5); log group contrast scale Normal(log .3,.7). Latents are noncentered Normal(0,1). Bronze/top are fixed references; champion baselines and group deviations use orthonormal zero-sum contrasts. Player and player–champion effects have proper zero-mean population distributions.

This extends the existing nonlinear C model with rank controls, champion baselines and group slopes. Both primary models use the **same exact Bernoulli NUTS inference**, rather than comparing an optimizer approximation with a sampler. No fitted ordering is forced to match A–J.

Convergence gates: every sampled parameter must have finite rank/folded split R-hat < 1.01 and bulk/tail ESS > 400; zero divergences; each chain acceptance ≥ .8. Tree-depth/BFMI are not emitted by this wrapper. Failed diagnostics save the chains/report and stop evaluation. A smoke run is always labeled smoke-only.

In [ ]:
display(pd.Series(modeling.model_specifications(), name='specification').to_frame())
print('Runtime settings:', CONFIG.sampling)
print('Both models fit only observed usable train rows; no validation/test refit.')
model_results = modeling.run_modeling(
    prepared, study_metadata, OUTPUT_PATH, CONFIG, enabled=RUN_MODELING)
if model_results is None:
    print('No model has been fitted in this run.')

## 8. Held-out evaluation, recommendations, and experience curves — populated only after fitting

After both full fits pass diagnostics, outputs include log loss, Brier score, ROC AUC, equal-player losses, calibration bins/plot, and row/player/match counts for identical validation/test rows. Predictions freeze training effects; full-patch rates make this **retrospective benchmark-assisted** evaluation.

Recommendation comparisons use final-test targets. They compare both the played pick and the existing all-role pregame main against eligible same-role choices with strictly higher **raw** matchup benchmarks. A main must itself have prior same-role support and a valid saved benchmark. An actual off-main pick may be a comparison alternative only if it also had prior same-role play. Every choice uses its own count, historical WR and group. Missing reference/alternative benchmarks do not become 50%.

**General** sets learned player and player–champion effects to zero; it does not integrate/marginalize their distributions. **Personalized** conditions on training-supported identity effects. New player/PC/champion identities draw from the appropriate pooled population distribution with uncertainty; identity-stable draws are reused across paired choices. Report paired 95% differences, posterior probability of a higher modeled win probability, uncertain preferences, and pair/equal-target/equal-player summaries. No unplayed outcome is observed or inferred as a label.

Group curves use a common Bronze/top context, training-median continuous features, and zero champion/player/PC baselines. They display posterior uncertainty, training games/players, and observed support; the plotted range stops at each group's training 95th percentile and cannot exceed the common reference total-games count. Rugs show actual observations inside that range; higher observations are not moved to the boundary. Groups need not follow A–J order. Curves vary by group, not independently by every champion. Sparse or single-player support should limit interpretation.

None of these comparisons establishes that switching champions would have caused a win, or that more practice caused the fitted association. Source benchmark error and the external study's transport assumptions are outside the conditional posterior intervals.

In [ ]:
if isinstance(model_results, dict) and 'metrics' in model_results:
    display(model_results['metrics'])
    display(model_results['recommendation_summary'])
    for name in ('shared_group_curves','grouped_group_curves','test_calibration'):
        display(Image(filename=str(OUTPUT_PATH / 'figures' / (name + '.png'))))
else:
    print('Evaluation/recommendations/curves are pending a converged full fit; no substantive model result is claimed.')
print('Output version:', OUTPUT_PATH)
for limitation in modeling.LIMITS:
    print('•', limitation)

## Running and resuming

First Run All with `RUN_MODELING=False` and review grouping, Unclassified support, history depths, missingness and splits. The workbook and cohort source stay pinned.

For an optional small execution check: install the optional backend in your notebook environment, set `profile='smoke'`, set `RUN_MODELING=True`, save, and Run All. This produces a separate smoke version; its small chains are not a convergence or performance result.

For the intended full comparison: restore `profile='full'`, enable the same switch, save, and Run All. Interrupting loses at most the current unfinished chain; reopening and Run All with the same sources/settings/code reuses the preparation version and verified completed chains. Full diagnostics must pass before performance or recommendations are exported. No package installation or collection happens automatically.